# Coherence diagnostics

**This is the check that would have caught the defect that shipped to five
repositories.** It was not a crash: the field looked plausible and was
wrong, so only numbers find it.

Three fingerprints of a coherence that is measuring signal *power* rather
than coupling:

1. cells pinned at exactly 1.0, hidden behind a clamp
2. a strong correlation between coherence and log power
3. no AR(1) null, so no individual value can be read at all

Run this whenever you recompute, and before you report anything.


In [ ]:
STUDY = '../../../case-demo'

import sys; sys.path.insert(0, '.')
from dims_checks.coherence import study_coherence_report, load_crosswavelet, coherence_report


## Every pair in the study


In [ ]:
import pandas as pd
rows = study_coherence_report(STUDY)
pd.DataFrame(rows)[['file', 'pair', 'saturated_at_1', 'corr_with_log_power',
                    'significant_fraction', 'source']]


## What looks wrong


In [ ]:
for r in rows:
    for w in r.get('warnings', []):
        print(f"{r.get('file')} :: {r.get('pair')}\n    {w}\n")


## Reading a single value

Coherence does **not** sit at zero when two signals are unrelated. It is a
ratio over a smoothing neighbourhood, so random phases still average to
around 0.25. That is why `sig95_wtc` exists: compare against the null, not
against your intuition.

`significant_fraction` near **0.05** means no coupling was detected —
however respectable the mean coherence looks.


In [ ]:
import glob, os, numpy as np, matplotlib.pyplot as plt
path = sorted(glob.glob(os.path.join(STUDY, 'assets/crosswavelet/*_data.json')))[0]
pairs, source = load_crosswavelet(path)
name, pair = next(iter(pairs.items()))
print(name, '·', source)
coherence_report(pair)


In [ ]:
coh, period, time = pair['coherence'], pair['period'], pair['time']
fig, ax = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
m = ax[0].pcolormesh(time, period, coh, vmin=0, vmax=1, shading='auto')
ax[0].set(yscale='log', title='coherence', xlabel='time (s)', ylabel='period (s)')
fig.colorbar(m, ax=ax[0])
null = pair.get('sig95_wtc')
if null is not None:
    ax[1].plot(period, null, label='95% AR(1) null')
    ax[1].plot(period, np.nanmean(coh, axis=1), label='mean coherence')
    ax[1].set(xscale='log', xlabel='period (s)', title='is it above the null?')
    ax[1].legend()
else:
    ax[1].text(.5, .5, 'no null in this file\nrecompute', ha='center')
plt.show()
